# Group 5 Capstone Starter Notebook

### 2024 Home Mortgage Disclosure Act (HMDA): Loan/Application Records (LAR)

- 1. Place raw data in `data/raw/`.
- 2. Save cleaned datasets in `data/processed/`.

Use this notebook for initial exploration, data checks, and quick experiments. Move reusable code into `src/` as the project matures.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# from google.colab import drive
import zipfile
import csv
from urllib.request import Request, urlopen
import shutil
import time
import ssl
import shutil
import subprocess
import time
from urllib.request import Request, urlopen

sns.set_theme(style="whitegrid")
# Mount Google Drive
# drive.mount('/content/drive')

## Download Helper Functions

In [2]:

# Define a function to download the zip file with retries and URL fallback
def download_data(url_link, colab_zipfile_path, zipfileName):
    """Download zip file using curl with retries and URL fallback."""

    colab_zipfile_path.mkdir(parents=True, exist_ok=True)
    download_target_path = colab_zipfile_path / zipfileName

    if download_target_path.exists():
        print(f"File {zipfileName} already exists at {download_target_path}. Skipping download.")
        return download_target_path

    if shutil.which("curl") is None:
        print("Error: curl is not installed or not available in PATH.")
        return None

    candidate_urls = [url_link]
    if "dynamic-data" in url_link:
        candidate_urls.append(url_link.replace("dynamic-data", "modified-lar"))

    user_agent = (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

    for candidate_url in candidate_urls:
        for attempt in range(1, 4):
            print(f"Downloading from {candidate_url} (attempt {attempt}/3)")
            cmd = [
                "curl",
                "-L",                 # follow redirects
                "--fail",             # non-2xx => non-zero exit
                "--silent",
                "--show-error",
                "--connect-timeout", "30",
                "--max-time", "300",
                "-A", user_agent,
                "-H", "Accept: */*",
                "-H", "Referer: https://ffiec.cfpb.gov/",
                "-o", str(download_target_path),
                candidate_url,
            ]

            result = subprocess.run(cmd, capture_output=True, text=True)

            if (
                result.returncode == 0
                and download_target_path.exists()
                and download_target_path.stat().st_size > 0
            ):
                print(f"Download complete: {download_target_path}")
                return download_target_path

            print(f"Attempt {attempt} failed: {result.stderr.strip() or 'Unknown curl error'}")
            if download_target_path.exists():
                download_target_path.unlink()
            time.sleep(2 ** (attempt - 1))

    print("Download failed after all retries and URL fallbacks.")
    return None

def getZipfile(RAW_DIR, zipfileName):
    # This function now correctly returns the Path object for the zip file in Google Drive
    return RAW_DIR / zipfileName


def unzpip_file(zip_path, extract_to):
    """Unzip a file to the specified directory."""
    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"File {zip_path} extracted successfully to {extract_to}")
    except Exception as e:
        print(f"Error extracting file: {e}")
        return None

def find_input_file(raw_dir, base_name="hmda_lar_2024"):
    """Find a likely HMDA LAR input file by name pattern and extension."""
    supported_ext = [".csv", ".txt"]

    # 1) Exact base-name match first
    for ext in supported_ext:
        candidate = raw_dir / f"{base_name}{ext}"
        if candidate.exists():
            return candidate

    # 2) Flexible match (handles names like 2024_lar.txt)
    candidates = [
        p for p in sorted(raw_dir.glob("*"))
        if p.is_file() and p.suffix.lower() in supported_ext
    ]

    def score_file(path_obj):
        name = path_obj.stem.lower()
        score = 0
        if "hmda" in name:
            score += 3
        if "lar" in name:
            score += 3
        if "2024" in name:
            score += 2
        if name == base_name.lower():
            score += 10
        return score

    ranked = sorted(candidates, key=score_file, reverse=True)
    if ranked and score_file(ranked[0]) > 0:
        return ranked[0]

    return None

def detect_delimiter(file_path):
    """Infer delimiter from extension, then validate using csv.Sniffer with fallback counting."""
    ext_default = {
        ".csv": ",",
        ".tsv": "\t",
        ".txt": "|",
        ".psv": "|",
    }
    delimiter = ext_default.get(file_path.suffix.lower(), ",")

    try:
        with open(file_path, "r", encoding="utf-8", newline="") as f:
            # Read only a sample to determine delimiter
            sample_lines = []
            for i, line in enumerate(f):
                sample_lines.append(line)
                if i >= 4: # Read first 5 lines
                    break
            sample = "".join(sample_lines)

        if sample.strip():
            try:
                dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", "|", ";"])
                delimiter = dialect.delimiter
            except Exception:
                # Fallback: choose delimiter with highest count in first non-empty line
                first_line = next((ln for ln in sample.splitlines() if ln.strip()), "")
                delimiter_counts = {
                    ",": first_line.count(","),
                    "\t": first_line.count("\t"),
                    "|": first_line.count("|"),
                    ";": first_line.count(";"),
                }
                best_delimiter = max(delimiter_counts, key=delimiter_counts.get)
                if delimiter_counts[best_delimiter] > 0:
                    delimiter = best_delimiter
    except Exception as e:
        print(f"Could not inspect delimiter, using '{delimiter}'. Reason: {e}")

    return delimiter

def load_data(file_path, delimiter='|'):
    """Load HMDA LAR data using the detected delimiter."""
    if not file_path.exists():
        print(f"Error: File {file_path} does not exist. Cannot load data.")
        return None
    try:
        print(f"Loading data from {file_path} with delimiter '{delimiter}'")
        df = pd.read_csv(file_path, sep=delimiter, low_memory=False)
        print(f"Data loaded successfully from {file_path} with delimiter '{delimiter}'")
        return df
    except Exception as e:
        print(f"Error loading data: {e}")
        return None


def main():

    # Define paths relative to the current working directory
    project_root = Path.cwd()
    if project_root.name == "notebooks":
        project_root = project_root.parent

    RAW_DIR = project_root / "data" / "raw"
    PROCESSED_DIR = project_root / "data" / "processed"

    # Create directories if they don't exist
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    # Define the base path in Google Drive for the zip file and its name
    colab_zipfile_path = Path("/data/raw")
    zipfileName = "2024_lar.zip"

    print(f"RAW_DIR: {RAW_DIR}")
    print(f"PROCESSED_DIR: {PROCESSED_DIR}")
    print(f"colab_zipfile_path: {colab_zipfile_path}")
    print(f"zipfileName: {zipfileName}")

    # Define URL for data download
    url_link = "https://files.ffiec.cfpb.gov/dynamic-data/2024/2024_lar.zip"

    # 1. Download the data from the URL to Google Drive if it doesn't exist
    downloader = globals().get("download_data")
    if downloader is None:
        raise NameError(
            "download_data is not defined. Ensure `def download_data(...)` is at top level (not indented inside another function)."
        )
    downloader(url_link, RAW_DIR, zipfileName)

    # 2. Get the full path to the downloaded zip file in Google Drive
    filepath = getZipfile(RAW_DIR, zipfileName)
    print(f"Zip file in Drive: {filepath}")

    # 3. Unzip the downloaded file from Google Drive to the local RAW_DIR
    # Check if the raw data file (expected after unzip) already exists in RAW_DIR
    raw_file_in_raw_dir = find_input_file(RAW_DIR, base_name="2024_lar")

    if filepath.exists():
        if raw_file_in_raw_dir:
            print(f"Extracted file {raw_file_in_raw_dir.name} already found in {RAW_DIR}. Skipping unzip.")
        else:
            print(f"Unzipping {filepath} to {RAW_DIR}")
            unzpip_file(filepath, RAW_DIR)
    else:
        print(f"Warning: Zip file not found at {filepath}. Cannot proceed with unzip or data processing.")
        print("Please ensure the download was successful or data is already extracted to RAW_DIR.")
        return # Exit if zip file not found

    # 4. Find source file and infer delimiter before reading from RAW_DIR
    raw_file = find_input_file(RAW_DIR, base_name="2024_lar")
    if raw_file is None:
        available = [p.name for p in sorted(RAW_DIR.glob("*")) if p.is_file()]
        print(f"No supported HMDA LAR file found in {RAW_DIR}.")
        print(f"Available files: {available}")
        print("Data loading or cleaning cannot proceed.")
        return
    else:
        print(f"Found input file: {raw_file}")

    print(f"Selected input file: {raw_file.name}")
    delimiter = '|'  # detect_delimiter(raw_file)
    print(f"Detected delimiter: {repr(delimiter)}")

In [3]:
# Define paths relative to the current working directory
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
print(f"Project root directory: {project_root}")
RAW_DIR = project_root / "data" / "raw"
print(f"RAW_DIR: {RAW_DIR}")
PROCESSED_DIR = project_root / "data" / "processed"
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

Project root directory: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project
RAW_DIR: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw
PROCESSED_DIR: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/processed


In [4]:
if __name__ == "__main__":
    main()

RAW_DIR: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw
PROCESSED_DIR: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/processed
colab_zipfile_path: /data/raw
zipfileName: 2024_lar.zip
Download complete: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw/2024_lar.zip
Zip file in Drive: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw/2024_lar.zip
Unzipping /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw/2024_lar.zip to /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw
File /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_CapStone_Final_Project/data/raw/